## Milvus Property Vector Search

End-to-end notebook for indexing and searching property listings in the Milvus collection **`property_listings`**.

**Embedding model:** `BAAI/bge-small-en-v1.5`  
**Vector dimension:** 384  
**Source CSV:** `data/processed/property_listings_raw.csv`

## Install dependencies

```bash
uv add fastembed pymilvus python-dotenv pandas
```

## 1. Imports and configuration

In [2]:
from pathlib import Path
import os
import math
import pandas as pd
from dotenv import load_dotenv
from fastembed import TextEmbedding
from pymilvus import MilvusClient, DataType

In [3]:
PROJECT_ROOT = Path.cwd().parent

CSV_PATH = PROJECT_ROOT / "data" / "processed" / "property_listings_raw.csv"
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH)

COLLECTION_NAME = "property_listings"
MODEL_NAME = "BAAI/bge-small-en-v1.5"
VECTOR_DIM = 384

MILVUS_URI = os.getenv("MILVUS_URI")
MILVUS_TOKEN = os.getenv("MILVUS_TOKEN")  

print("Project root:", PROJECT_ROOT.resolve())
print("CSV exists:", CSV_PATH.exists())
print("Collection:", COLLECTION_NAME)
print("Embedding model:", MODEL_NAME)
print("MILVUS_URI configured:", bool(MILVUS_URI))

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {CSV_PATH}. Run the notebook from the project root "
        "or update PROJECT_ROOT / CSV_PATH."
    )

if not MILVUS_URI:
    raise ValueError(
        "MILVUS_URI is missing. Add it to your project's .env file. "
        "Do not paste your token or credentials into notebook cells."
    )

Project root: /home/momina/projects/Real-estate-research-agent
CSV exists: True
Collection: property_listings
Embedding model: BAAI/bge-small-en-v1.5
MILVUS_URI configured: True


## 2. Connect to Milvus

In [4]:
connection_args = {"uri": MILVUS_URI}
if MILVUS_TOKEN:
    connection_args["token"] = MILVUS_TOKEN

client = MilvusClient(**connection_args)

print("Connected to Milvus.")
print("Collection exists:", client.has_collection(collection_name=COLLECTION_NAME))
print("Collections visible to this connection:", client.list_collections())

Connected to Milvus.
Collection exists: True
Collections visible to this connection: ['property_listings']


## 3. Load and inspect the raw CSV

The raw CSV remains unchanged. We create a working DataFrame for the vector index.

In [5]:
df = pd.read_csv(CSV_PATH, low_memory=False)

print("Shape:", df.shape)
display(df.head(3))

expected_columns = [
    "id", "title", "price", "beds", "bathrooms", "community",
    "property_type", "listing_type", "status", "description",
    "long_description", "link"
]
print("Missing expected columns:", [c for c in expected_columns if c not in df.columns])

Shape: (6495, 57)


,id,crm_id,strapi_id,slug,title,listing_type,department,search_type,status,price,...,agent_phone,agent_whatsapp,agent_designation,agent_languages,agent_photo_url,agent_slug,video_tour,virtual_tour,enriched_at,property_type
0,9971,DBI215069,9971,2-bedroom-apartment-for-sale-in-mosela-watersi...,2 bedroom apartment for sale,purchase,residential,sales,For Sale,2850000.0,...,+971 58 577 0306,+971 58 577 0306,Greens & Views Consultant,"[""English"",""Turkish""]",https://ggfx-handh3.s3.eu-west-2.amazonaws.com...,murat-topcuoglu,[],[],2026-09-17T17:03:40Z,apartment
1,12292,DUB203155,12292,3-bedroom-villa-for-sale-in-district-8-jumeira...,3 bedroom villa for sale,purchase,residential,sales,For Sale,7195000.0,...,+971 50 598 0269,+971 50 598 0269,Associate Partner,"[""English""]",https://ggfx-handh3.s3.eu-west-2.amazonaws.com...,cathal-o-neill,[],[],2026-09-17T18:04:47Z,villa
2,10051,DBI210323,10051,2-bedroom-apartment-for-sale-in-al-ghozlan-1-t...,2 bedroom apartment for sale,purchase,residential,sales,For Sale,2250000.0,...,+971 58 577 0306,+971 58 577 0306,Greens & Views Consultant,"[""English"",""Turkish""]",https://ggfx-handh3.s3.eu-west-2.amazonaws.com...,murat-topcuoglu,[],[],2026-09-17T18:04:47Z,apartment


Missing expected columns: []


In [6]:
# Basic data-quality checks
print("Rows:", len(df))
if "id" in df.columns:
    print("Missing property IDs:", df["id"].isna().sum())
    print("Duplicate property IDs:", df["id"].astype("string").duplicated().sum())

for col in ["price", "beds", "bathrooms"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

display(df[["price", "beds", "bathrooms"]].describe().T)

Rows: 6495
Missing property IDs: 0
Duplicate property IDs: 0


,count,mean,std,min,25%,50%,75%,max
price,6495.0,1.739161e+06,6.242300e+06,0.0,115000.0,200000.0,875000.0,249000000.0
beds,6495.0,2.236182e+00,1.458580e+00,0.0,1.0,2.0,3.0,9.0
bathrooms,6495.0,2.812779e+00,1.603351e+00,0.0,2.0,3.0,4.0,16.0


## 4. Prepare property text and metadata

Semantic embeddings are for meaning and relevance. Exact constraints such as price, bedroom count, listing type and status should be applied as metadata filters, not left to the embedding model.

In [7]:
def clean_text(value):
    if pd.isna(value):
        return ""
    # Basic HTML tag removal and whitespace normalization.
    import re
    value = str(value)
    value = re.sub(r"<[^>]+>", " ", value)
    value = re.sub(r"&nbsp;|&#160;", " ", value, flags=re.IGNORECASE)
    value = re.sub(r"\s+", " ", value)
    return value.strip()

def text_value(row, column):
    return clean_text(row[column]) if column in row.index else ""

def make_search_text(row):
    parts = [
        f"Title: {text_value(row, 'title')}",
        f"Property type: {text_value(row, 'property_type')}",
        f"Community: {text_value(row, 'community')}",
        f"Building: {text_value(row, 'building')}",
        f"Bedrooms: {text_value(row, 'beds')}",
        f"Bathrooms: {text_value(row, 'bathrooms')}",
        f"Description: {text_value(row, 'description')}",
        f"Additional details: {text_value(row, 'accommodation_summary')}",
        f"Long description: {text_value(row, 'long_description')}",
    ]
    return " ".join(part for part in parts if part.split(": ", 1)[-1].strip())

# Stable property ID is required for deduplication and repeatable indexing
df["property_id"] = df["id"].astype("string").str.strip()
df = df[df["property_id"].notna() & (df["property_id"] != "")].copy()
df = df.drop_duplicates(subset=["property_id"], keep="last").copy()

df["search_text"] = df.apply(make_search_text, axis=1)
df = df[df["search_text"].str.strip().ne("")].copy()

print("Rows eligible for indexing:", len(df))
display(df[["property_id", "title", "community", "price", "beds", "search_text"]].head(3))

Rows eligible for indexing: 6495


,property_id,title,community,price,beds,search_text
0,9971,2 bedroom apartment for sale,The Views,2850000.0,2,Title: 2 bedroom apartment for sale Property t...
1,12292,3 bedroom villa for sale,Jumeirah Park,7195000.0,3,Title: 3 bedroom villa for sale Property type:...
2,10051,2 bedroom apartment for sale,The Greens,2250000.0,2,Title: 2 bedroom apartment for sale Property t...


## 5. Initialize the embedding model and test one vector

The BGE query instruction is used for **search queries**. For property documents, embed the prepared property text without that query prefix.

In [8]:
embedding_model = TextEmbedding(model_name=MODEL_NAME)

sample_text = df.iloc[0]["search_text"]
sample_vector = list(embedding_model.embed([sample_text]))[0]

print("Vector dimensions:", len(sample_vector))
print("First 5 vector values:", sample_vector[:5])

assert len(sample_vector) == VECTOR_DIM, (
    f"Expected {VECTOR_DIM} dimensions, got {len(sample_vector)}"
)

Vector dimensions: 384
First 5 vector values: [ 0.00575617  0.03332201 -0.01086792  0.05088252  0.04747245]


## 6. Define the Milvus schema

Schema fields:
- `property_id`: stable primary key from the source `id`
- `embedding`: 384-dimensional vector
- typed scalar metadata used for filters
- short text fields returned in search results

In [8]:
def build_collection_schema():
    schema = client.create_schema(auto_id=False, enable_dynamic_field=False)

    schema.add_field(
        field_name="property_id",
        datatype=DataType.VARCHAR,
        is_primary=True,
        max_length=128,
    )
    schema.add_field(
        field_name="embedding",
        datatype=DataType.FLOAT_VECTOR,
        dim=VECTOR_DIM,
    )
    schema.add_field(field_name="price", datatype=DataType.DOUBLE)
    schema.add_field(field_name="beds", datatype=DataType.INT64)
    schema.add_field(field_name="bathrooms", datatype=DataType.DOUBLE)
    schema.add_field(field_name="community", datatype=DataType.VARCHAR, max_length=512)
    schema.add_field(field_name="property_type", datatype=DataType.VARCHAR, max_length=256)
    schema.add_field(field_name="listing_type", datatype=DataType.VARCHAR, max_length=128)
    schema.add_field(field_name="status", datatype=DataType.VARCHAR, max_length=128)
    schema.add_field(field_name="title", datatype=DataType.VARCHAR, max_length=1024)
    schema.add_field(field_name="link", datatype=DataType.VARCHAR, max_length=2048)
    schema.add_field(field_name="search_text", datatype=DataType.VARCHAR, max_length=8192)
    return schema

if client.has_collection(collection_name=COLLECTION_NAME):
    existing_description = client.describe_collection(collection_name=COLLECTION_NAME)
    print("Collection already exists; not recreating it.")
    display(existing_description)
    print("\nInspect the fields above. The expected schema is shown in the previous markdown cell.")
else:
    schema = build_collection_schema()
    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name="embedding",
        index_type="AUTOINDEX",
        metric_type="COSINE",
    )
    client.create_collection(
        collection_name=COLLECTION_NAME,
        schema=schema,
        index_params=index_params,
    )
    print(f"Created collection: {COLLECTION_NAME}")

print("Collection ready check:", client.has_collection(collection_name=COLLECTION_NAME))

Created collection: property_listings
Collection ready check: True


## 7. Validate the collection schema before writing

In [9]:
description = client.describe_collection(collection_name=COLLECTION_NAME)
fields_info = description.get("fields", [])
field_map = {field.get("name"): field for field in fields_info}

print("Field names:", list(field_map.keys()))

required_fields = {
    "property_id", "embedding", "price", "beds", "bathrooms",
    "community", "property_type", "listing_type", "status",
    "title", "link", "search_text"
}
missing_fields = required_fields - set(field_map)

if missing_fields:
    raise RuntimeError(
        f"Existing collection '{COLLECTION_NAME}' is missing required fields: "
        f"{sorted(missing_fields)}. Do not drop it automatically. Inspect the schema "
        "and choose a migration or a separately named test collection."
    )

vector_field = field_map["embedding"]
vector_params = vector_field.get("params", {})
if vector_params and int(vector_params.get("dim", VECTOR_DIM)) != VECTOR_DIM:
    raise RuntimeError(
        f"Collection vector dimension is incompatible: {vector_params.get('dim')} "
        f"(expected {VECTOR_DIM}). Do not drop it automatically."
    )

print("Required fields appear present.")

Field names: ['property_id', 'embedding', 'price', 'beds', 'bathrooms', 'community', 'property_type', 'listing_type', 'status', 'title', 'link', 'search_text']
Required fields appear present.


## 8. Convert rows into Milvus records

Missing numeric values are represented as `0` for price/beds/bathrooms in this first version. For business-critical filtering, treat these values as unknown and exclude them from matches requiring that field. Strings are trimmed to fit schema limits.

In [10]:
def safe_number(value, default=0.0):
    try:
        if pd.isna(value):
            return default
        value = float(value)
        if not math.isfinite(value):
            return default
        return value
    except (TypeError, ValueError):
        return default

def safe_int(value, default=0):
    number = safe_number(value, default)
    return int(number) if number >= 0 else default

def safe_string(value, max_length, default=""):
    value = clean_text(value)
    return (value or default)[:max_length]

def row_to_record(row, vector):
    price = safe_number(row.get("price"), 0.0)
    beds = safe_int(row.get("beds"), 0)
    bathrooms = safe_number(row.get("bathrooms"), 0.0)

    return {
        "property_id": str(row["property_id"])[:128],
        "embedding": vector.tolist() if hasattr(vector, "tolist") else list(vector),
        "price": price,
        "beds": beds,
        "bathrooms": bathrooms,
        "community": safe_string(row.get("community"), 512),
        "property_type": safe_string(row.get("property_type"), 256),
        "listing_type": safe_string(row.get("listing_type"), 128),
        "status": safe_string(row.get("status"), 128),
        "title": safe_string(row.get("title"), 1024),
        "link": safe_string(row.get("link"), 2048),
        "search_text": safe_string(row.get("search_text"), 8192),
    }

## 9. Embed the properties in batches

This can take a few minutes on the first run. Batches keep memory usage manageable. This cell prepares records in memory; the next cell writes them to Milvus.

In [11]:
BATCH_SIZE = 128
records = []
texts = df["search_text"].tolist()
rows = [row for _, row in df.iterrows()]

for start in range(0, len(texts), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(texts))
    vectors = list(embedding_model.embed(texts[start:end]))
    for row, vector in zip(rows[start:end], vectors):
        records.append(row_to_record(row, vector))
    if start == 0 or end == len(texts) or end % (BATCH_SIZE * 5) == 0:
        print(f"Embedded {end:,} / {len(texts):,}")

print("Prepared records:", len(records))
print("Vector dimension:", len(records[0]["embedding"]) if records else "no records")

Embedded 128 / 6,495
Embedded 640 / 6,495
Embedded 1,280 / 6,495
Embedded 1,920 / 6,495
Embedded 2,560 / 6,495
Embedded 3,200 / 6,495
Embedded 3,840 / 6,495
Embedded 4,480 / 6,495
Embedded 5,120 / 6,495
Embedded 5,760 / 6,495
Embedded 6,400 / 6,495
Embedded 6,495 / 6,495
Prepared records: 6495
Vector dimension: 384


## 10. Insert or safely upsert records

Milvus `upsert` inserts new primary keys and updates records with matching primary keys. It is convenient for repeat runs when source IDs are stable.

In [13]:
WRITE_BATCH_SIZE = 128
written = 0

for start in range(0, len(records), WRITE_BATCH_SIZE):
    batch = records[start:start + WRITE_BATCH_SIZE]
    result = client.upsert(
        collection_name=COLLECTION_NAME,
        data=batch,
    )
    written += len(batch)
    print(f"Upserted {written:,} / {len(records):,}", "| result:", result)

client.flush(collection_name=COLLECTION_NAME)
print("Write complete.")

Upserted 128 / 6,495 | result: {'upsert_count': 128, 'cost': 232, 'ids': ['9971', '12292', '10051', '6452', '5191', '19094', '20037', '16266', '298', '13947', '10647', '21148', '21371', '21348', '5963', '6385', '10290', '10155', '21666', '16406', '20813', '22737', '22956', '23668', '24405', '25282', '29227', '24705', '25833', '26007', '29776', '28317', '29453', '30672', '26929', '28625', '30531', '29959', '31297', '31836', '31689', '37377', '31971', '35706', '37295', '32822', '33213', '41559', '37984', '32667', '37553', '41217', '40883', '31616', '42340', '42638', '42770', '44184', '43419', '47936', '47444', '49582', '48762', '49502', '47937', '47673', '49597', '50914', '51519', '52039', '55363', '58627', '58748', '59124', '60130', '58289', '60740', '62319', '60418', '60986', '66147', '60186', '64679', '62212', '66590', '64174', '67761', '58116', '68565', '67072', '68445', '68855', '71411', '71878', '71026', '70682', '76305', '76667', '76038', '75463', '68948', '70253', '77359', '78855

## 11. Load the collection and verify the index

In [14]:
client.load_collection(collection_name=COLLECTION_NAME)
print("Collection loaded.")

stats = client.get_collection_stats(collection_name=COLLECTION_NAME)
display(stats)

Collection loaded.


{'row_count': 6495}

## 12. Semantic search

Use the BGE retrieval instruction on the **query**. Results are ranked by cosine similarity.

In [15]:
def embed_query(query):
    query_for_embedding = (
        "Represent this sentence for searching relevant passages: " + query.strip()
    )
    vector = list(embedding_model.embed([query_for_embedding]))[0]
    return vector.tolist() if hasattr(vector, "tolist") else list(vector)

def search_properties(query, limit=10, filter_expr=None):
    query_vector = embed_query(query)
    results = client.search(
        collection_name=COLLECTION_NAME,
        data=[query_vector],
        anns_field="embedding",
        limit=limit,
        filter=filter_expr,
        output_fields=[
            "property_id", "title", "price", "beds", "bathrooms",
            "community", "property_type", "listing_type", "status", "link"
        ],
        search_params={"metric_type": "COSINE", "params": {}},
    )

    hits = results[0] if results else []
    output = []
    for hit in hits:
        entity = hit.get("entity", {})
        output.append({
            "property_id": entity.get("property_id"),
            "title": entity.get("title"),
            "price": entity.get("price"),
            "beds": entity.get("beds"),
            "bathrooms": entity.get("bathrooms"),
            "community": entity.get("community"),
            "property_type": entity.get("property_type"),
            "listing_type": entity.get("listing_type"),
            "status": entity.get("status"),
            "link": entity.get("link"),
            "similarity_score": hit.get("distance"),
        })
    return pd.DataFrame(output)

query = "A spacious family home close to parks and family-friendly amenities"
results_df = search_properties(query, limit=10)
display(results_df)

,property_id,title,price,beds,bathrooms,community,property_type,listing_type,status,link,similarity_score
0,14642,4 bedroom villa to rent,324999.0,4,5.0,Green Community,villa,rental,To Let,https://www.hausandhaus.com/properties/to-rent...,0.699423
1,65662,5 bedroom villa to rent,340000.0,5,6.0,Dubai Investment Park,villa,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.689298
2,4241,5 bedroom villa to rent,220000.0,5,6.0,Green Community,villa,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.687948
3,3153,5 bedroom villa to rent,350000.0,5,5.0,Dubai Investment Park,villa,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.687105
4,2805,5 bedroom villa to rent,210000.0,5,5.0,Green Community,villa,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.681967
5,3093,4 bedroom villa to rent,379999.0,4,5.0,Jumeirah Park,villa,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.680766
6,65707,5 bedroom villa to rent,350000.0,5,7.0,Dubai Investment Park,villa,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.680359
7,83789,4 bedroom villa to rent,250000.0,4,4.0,Mudon,villa,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.680062
8,4026,4 bedroom villa to rent,290000.0,4,5.0,Dubai Investment Park,villa,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.679083
9,131629,4 bedroom villa to rent,360000.0,4,4.0,Jumeirah Park,villa,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.679036


## 13. Hard-filtered search examples

Milvus filter expressions use the exact field names in the collection schema. Check the dataset's actual `listing_type` and `status` values before relying on them in production.

In [16]:
# Example: two-bedroom properties under AED 2,000,000.
# price > 0 excludes records where price was missing/zero in the source.
filter_expr = 'price > 0 and price <= 2000000 and beds == 2'

filtered_results = search_properties(
    "A modern apartment suitable for a small family",
    limit=10,
    filter_expr=filter_expr,
)
display(filtered_results)

,property_id,title,price,beds,bathrooms,community,property_type,listing_type,status,link,similarity_score
0,14647,2 bedroom apartment to rent,135000.0,2,3.0,Green Community,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.703047
1,61833,2 bedroom apartment to rent,145000.0,2,3.0,Downtown Dubai,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.697495
2,146074,2 bedroom apartment to rent,120000.0,2,2.0,Al Sufouh,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.692053
3,2404,2 bedroom apartment to rent,110000.0,2,3.0,Green Community,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.691492
4,100777,2 bedroom apartment to rent,190000.0,2,3.0,Downtown Dubai,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.682222
5,3303,2 bedroom apartment to rent,156000.0,2,3.0,Jumeirah Golf Estates,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.680774
6,88460,2 bedroom apartment to rent,250000.0,2,3.0,Mohammed Bin Rashid City,apartment,rental,To Let,https://www.hausandhaus.com/properties/to-rent...,0.679457
7,122771,2 bedroom apartment to rent,190000.0,2,3.0,Greens,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.678847
8,2731,2 bedroom apartment to rent,120000.0,2,2.0,The Greens,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.676869
9,94590,2 bedroom apartment to rent,155000.0,2,3.0,Downtown Dubai,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.676349


In [17]:
# Example: filter by community. Escape double quotes in the user-supplied value.
def escape_milvus_string(value):
    return str(value).replace("\\", "\\\\").replace('"', '\\"')

community_name = "Dubai Marina"
filter_expr = (
    f'price > 0 and community like "%{escape_milvus_string(community_name)}%"'
)

community_results = search_properties(
    "Apartment with a sea view and convenient local amenities",
    limit=10,
    filter_expr=filter_expr,
)
display(community_results)

,property_id,title,price,beds,bathrooms,community,property_type,listing_type,status,link,similarity_score
0,2962,1 bedroom apartment to rent,75000.0,1,2.0,Dubai Marina,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.748335
1,59200,1 bedroom apartment to rent,105000.0,1,1.0,Dubai Marina,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.745598
2,3141,2 bedroom apartment to rent,100000.0,2,3.0,Dubai Marina,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.745192
3,4480,1 bedroom apartment to rent,105000.0,1,2.0,Dubai Marina,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.741888
4,4362,2 bedroom apartment to rent,115000.0,2,1.0,Dubai Marina,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.741868
5,4383,1 bedroom apartment to rent,119000.0,1,2.0,Dubai Marina,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.740659
6,50906,1 bedroom apartment to rent,125000.0,1,1.0,Dubai Marina,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.738273
7,2963,2 bedroom apartment to rent,95000.0,2,3.0,Dubai Marina,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.736571
8,3821,2 bedroom apartment to rent,105000.0,2,2.0,Dubai Marina,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.734023
9,4390,2 bedroom apartment to rent,175000.0,2,3.0,Dubai Marina,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,0.733214


## 14. Inspect distinct metadata values

Use this before deciding which status or listing-type values should represent active sale/rent inventory. Do not assume `status` semantics without inspecting the source.

In [18]:
for col in ["listing_type", "status", "property_type"]:
    if col in df.columns:
        print(f"\n{col} — top values")
        display(df[col].astype("string").fillna("<missing>").value_counts().head(20).rename_axis(col).reset_index(name="count"))


listing_type — top values


,listing_type,count
0,rental,4973
1,purchase,1522



status — top values


,status,count
0,Let,3529
1,For Sale,1454
2,To Let,1444
3,Sold,68



property_type — top values


,property_type,count
0,apartment,3904
1,villa,1717
2,townhouse,556
3,office space,252
4,retail,37
5,<missing>,18
6,land,5
7,warehouse,4
8,whole building,1
9,labour camp,1


## 15. Retrieve a property by its ID

This uses the primary key for an exact lookup, not semantic similarity.

In [19]:
def get_property(property_id):
    rows = client.get(
        collection_name=COLLECTION_NAME,
        ids=[str(property_id)],
        output_fields=[
            "property_id", "title", "price", "beds", "bathrooms",
            "community", "property_type", "listing_type", "status", "link", "search_text"
        ],
    )
    return pd.DataFrame(rows)

# Example:
display(get_property("4383"))

,community,property_id,price,beds,property_type,listing_type,status,link,search_text,title,bathrooms
0,Dubai Marina,4383,119000.0,1,apartment,rental,Let,https://www.hausandhaus.com/properties/to-rent...,Title: 1 bedroom apartment to rent Property ty...,1 bedroom apartment to rent,2.0
